<a href="https://colab.research.google.com/github/MukulVerma-ML/Titanic-Survival-Predictor/blob/main/notebooks/Day_5_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [70]:
train=pd.read_csv('train[1].csv')
test=pd.read_csv('test[1].csv')
test_ids=test['PassengerId']

df=pd.concat([train,test],sort=False).reset_index(drop=True)


In [51]:
df['Age']= df['Age'].fillna(df['Age'].mean())
df['Fare']= df['Fare'].fillna(df['Fare'].mean())
df['Embarked']= df['Embarked'].fillna(df['Embarked'].mode()[0])

In [52]:
df['FamilySize']= df['SibSp'] + df['Parch'] +1
df['IsAlone']=0
df.loc[df['FamilySize']==1,'IsAlone']=1

df['Title'] = df['Name'].str.extract(r'([A-Za-z]+)\.',expand=False)
df['Title'] = df['Title'].replace(['Lady','Countess','Capt','Col','Don','Doctor','Major','Rev','Sir','Johnkheer','Dona'],'Rare')
df['Title'] = df['Title'].replace(['Mlle','Ms'],'Miss')
df['Title'] = df['Title'].replace('Mme','Mrs')

In [53]:
df['Deck'] = df['Cabin'].str[0]
df['Deck'] =df['Deck'].fillna('U')
df['Deck'] =df['Deck'].map({'A':1,'B':2,'C':3,'D':4,'E':5,'F':6,'G':7,'T':8,'U':0})

In [54]:
df['HasCabin'] =df['Cabin'].notnull().astype(int)

In [55]:
df['TicketGroup'] = df.groupby('Ticket')['Ticket'].transform('count')

In [56]:
df['IsAlone_Ticket']=0
df.loc[df['TicketGroup']==1,'IsAlone']=1

In [57]:
df['TicketLetter'] = df['Ticket'].str.contains('[A-Za-z]').astype(int)

In [58]:
le=LabelEncoder()
df['Sex'] = le.fit_transform(df['Sex'])
df['Embarked'] = le.fit_transform(df['Embarked'])
df['Title'] = le.fit_transform(df['Title'])

In [64]:
df.drop(['Ttitle','Name','Cabin','Ticket','Passenger','SibSp','Parch'],axis=1,inplace=True,errors='ignore')

train_df =df[df['Survived'].notnull()]
test_df =df[df['Survived'].isnull()].drop('Survived',axis=1)

X = train_df.drop('Survived', axis=1)
Y = train_df['Survived']

X_train,X_val,Y_train,Y_val=train_test_split(X,Y,test_size=0.2,random_state=42)

model = RandomForestClassifier(n_estimators=100,random_state=42)
print(X_train.dtypes)

model.fit(X_train,Y_train)

val_acc = model.score(X_val,Y_val)
print(f"Validation Accuracy: {val_acc:4f}")


PassengerId         int64
Pclass              int64
Sex                 int64
Age               float64
Fare              float64
Embarked            int64
FamilySize          int64
IsAlone             int64
Title               int64
Deck                int64
HasCabin            int64
TicketGroup         int64
IsAlone_Ticket      int64
TicketLetter        int64
dtype: object
Validation Accuracy: 0.854749


In [65]:
feat_imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop Features:")
print(feat_imp.head(8))



Top Features:
Sex            0.179621
PassengerId    0.151968
Age            0.144257
Fare           0.143907
Title          0.098491
Pclass         0.056004
TicketGroup    0.055447
FamilySize     0.044373
dtype: float64


In [72]:
test_pred = model.predict(test_df)
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': test_pred.astype(int)
})
submission.to_csv('submission_day5.csv', index=False)

print("\nDay 4 Score: 0.780")
print(f"Day 5 Score: {val_acc:.3f}")



Day 4 Score: 0.780
Day 5 Score: 0.855
